# Exercise 5: Classification, Evaluation, and Pipelines

Exercise 4 predicted a number, the value of a house. This exercise predicts a category: whether a tumour is malignant, and whether a loan applicant will repay. A model that predicts a category is a **classifier**, and the task is **classification**. The response takes one of a few values, the **classes**, and a classifier usually gives a probability for each class as well as a decision.

A classifier is judged differently from a regressor. An error is either a missed case or a false alarm, the two can cost very different amounts, one class can be much rarer than the other, and a probability becomes a decision only at a chosen **threshold**. The exercise therefore spends as much time on evaluation as on the models.

The exercise has three parts:

- **classifiers** on the breast cancer data in `data/breast-cancer.csv`: whether each of 569 tumours is malignant, predicted with logistic regression, nearest neighbours, a decision tree, and naive Bayes, scored with the confusion matrix and its metrics, cross-validated in pipelines, and tuned,
- **naive Bayes** worked by hand on the ten records of `data/tax-evasion.csv`, and
- **evaluation** on the credit data in `data/german-credit.csv`: 1,000 loan applicants, used for thresholds, curves, and class imbalance.

## Setup

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE, RandomOverSampler
from imblearn.pipeline import make_pipeline as make_imbalanced_pipeline
from imblearn.under_sampling import RandomUnderSampler
from scipy import stats
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from sklearn.model_selection import (GridSearchCV, GroupKFold, RandomizedSearchCV, RepeatedStratifiedKFold,
                                     StratifiedKFold, TimeSeriesSplit, cross_val_predict, cross_val_score,
                                     cross_validate, train_test_split, validation_curve)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

DATA = Path("data")
SEED = 2026
# A model that never flags the positive class has no precision, and scikit-learn reports 0.
warnings.filterwarnings("ignore", message="Precision is ill-defined")

In [ ]:
# The course style. The colours repeat template/themes/data-science.css, so a change there is repeated here.
PALETTE = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#1e2a24"]
GROUND, INK, GRID = "#ffffff", "#1e2a24", "#e2e6e4"
plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.facecolor": GROUND,
    "figure.constrained_layout.use": True,
    "savefig.facecolor": GROUND,
    "savefig.transparent": False,
    "font.family": "sans-serif",
    "font.sans-serif": ["Archivo", "DejaVu Sans"],
    "font.size": 12,
    "text.color": INK,
    "axes.facecolor": GROUND,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.axisbelow": True,
    "axes.prop_cycle": plt.cycler(color=PALETTE),
    "grid.color": GRID,
    "xtick.color": INK,
    "ytick.color": INK,
    "legend.frameon": False,
    "image.cmap": "cividis",
})
sns.set_palette(PALETTE)

## 1. Classification and the Tumour Data

In **classification**, the response is a category.

Each row of `data/breast-cancer.csv` is a tumour, described by 30 measurements of the cell nuclei in an image of a fine-needle biopsy, such as their mean radius and the smoothness of their outline, and `diagnosis` records whether the tumour is malignant or benign.

In [ ]:
cancer = pd.read_csv(DATA / "breast-cancer.csv")
cancer.shape, cancer["diagnosis"].value_counts()

In [ ]:
cancer.sample()

A malignant tumour is the case to detect, so it is coded as 1, the **positive class**, and a benign tumour as 0, the **negative class**. The words describe the class, not good or bad news. The split is **stratified** by the diagnosis: `stratify=y` keeps the share of malignant tumours the same in the training and test sets, which matters when one class is rarer than the other.

In [ ]:
X_cells = cancer.drop(columns="diagnosis")
y_cells = (cancer["diagnosis"] == "malignant").astype(int)
X_fit, X_held, y_fit, y_held = train_test_split(X_cells, y_cells, test_size=0.2, stratify=y_cells,
                                                random_state=SEED)
len(X_fit), len(X_held), round(y_fit.mean(), 3), round(y_held.mean(), 3)

The 114 held-out tumours are the test set of this part. Every choice is made on the 455 training tumours, by cross-validation.

## 2. Logistic Regression

Linear regression on a 0 or 1 response would predict values below 0 and above 1, which no probability can take. **Logistic regression** models the probability of the positive class directly, by passing the linear combination of the predictors through the S-shaped **sigmoid** function, which squeezes any number into the interval from 0 to 1:

$$p(x) = P(y = 1 \mid x) = \frac{1}{1 + e^{-(\beta_0 + \beta_1 x_1 + \dots + \beta_p x_p)}}$$

A few values show how the sigmoid squeezes: a linear combination of 0 gives a probability of exactly 0.5, a value of 2 gives 0.88, a value of 4 gives 0.98, and negative values mirror these below 0.5.

In [ ]:
log_odds_values = np.array([-4, -2, 0, 2, 4])
pd.Series(1 / (1 + np.exp(-log_odds_values)), index=log_odds_values).round(3)

The same model says that the **log-odds** are linear in the predictors: $\log \frac{p}{1 - p} = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$. The **odds** $p / (1 - p)$ compare the chance of the positive class with the chance of the negative one: $p = 0.2$ gives odds of 0.25, and $p = 0.9$ gives odds of 9. A one-unit increase of $x_j$ therefore multiplies the odds by $e^{\beta_j}$.

The coefficients are chosen by **maximum likelihood**, which minimizes the **log loss**:

$$\text{log loss} = -\frac{1}{n}\sum_i \big[y_i \log p_i + (1 - y_i) \log(1 - p_i)\big]$$

A confident wrong prediction costs a lot: a malignant tumour given $p = 0.1$ adds $-\log 0.1 = 2.3$, while one given $p = 0.9$ adds only 0.1. There is no closed form, so the coefficients are found step by step. Scikit-learn's `LogisticRegression` also adds a ridge penalty by default, whose strength is set by `C`, the inverse of $\lambda$: a smaller `C` means a stronger penalty. The predictors are standardized first.

In [ ]:
logistic = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_fit, y_fit)
odds = pd.Series(np.exp(logistic[-1].coef_[0]), index=X_cells.columns)
odds.sort_values().iloc[[0, 1, -2, -1]].round(2)

Each additional standard deviation of `worst texture` multiplies the odds of malignancy by more than 3, with the other measurements held fixed. For a tumour whose odds were 1 to 1, a probability of 0.5, one standard deviation more texture makes them 3.34 to 1, a probability of $3.34 / 4.34 = 0.77$. An odds ratio below 1, such as 0.53 for `fractal dimension error`, lowers the odds instead. `predict_proba` returns the probability of each class, and `predict` returns the class whose probability is at least 0.5.

In [ ]:
probabilities = logistic.predict_proba(X_held)[:, 1]
pd.DataFrame({"P(malignant)": probabilities[:6].round(3), "predicted": logistic.predict(X_held)[:6],
              "actual": y_held.to_numpy()[:6]})

In [ ]:
log_odds = logistic.decision_function(X_held)
order = np.argsort(log_odds)
colours = np.where(y_held.to_numpy()[order] == 1, PALETTE[1], PALETTE[0])
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.scatter(log_odds[order], probabilities[order], c=colours, s=18)
ax.axhline(0.5, color=INK, linestyle="--")
ax.set(xlabel="Linear combination of the predictors (log-odds)", ylabel="P(malignant)",
       title="The sigmoid turns log-odds into a probability (orange: malignant)")
plt.show()

## 3. The Confusion Matrix and Its Metrics

A classifier is scored by counting its decisions. The **confusion matrix** crosses the actual class with the predicted one, and its four cells name the four outcomes:

- a **true positive (TP)** is a malignant tumour predicted malignant,
- a **false negative (FN)** is a malignant tumour predicted benign, a missed cancer,
- a **false positive (FP)** is a benign tumour predicted malignant, a false alarm, and
- a **true negative (TN)** is a benign tumour predicted benign.

In [ ]:
predicted_classes = logistic.predict(X_held)
confusion_matrix(y_held, predicted_classes)

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay.from_predictions(y_held, predicted_classes, display_labels=["benign", "malignant"],
                                        cmap="cividis", colorbar=False, ax=ax)
ax.set(title="Logistic regression, 114 test tumours")
ax.grid(False)
plt.show()

Scikit-learn puts the actual class in the rows and the predicted class in the columns, with the negative class first. Read row by row: of the 72 benign tumours, 71 were predicted benign and 1 malignant, a false alarm; of the 42 malignant tumours, 40 were found and 2 were missed. Every metric is computed from these four counts:

- **accuracy** $= (TP + TN) / n$, the share of correct decisions,
- **precision** $= TP / (TP + FP)$, the share of the tumours flagged malignant that are malignant, which measures how reliable a flag is,
- **recall** $= TP / (TP + FN)$, also called sensitivity, the share of the malignant tumours that are flagged, which measures how few cancers are missed, and
- the **F1 score** $= 2 \cdot \text{precision} \cdot \text{recall} / (\text{precision} + \text{recall})$, their harmonic mean, which is high only when both are high.

With the counts above, the accuracy is $(40 + 71) / 114 = 0.974$, the precision is $40 / 41 = 0.976$, and the recall is $40 / 42 = 0.952$. The harmonic mean matters when the two disagree: a model with a precision of 1.0 and a recall of 0.1 has an arithmetic mean of 0.55 but an F1 of only 0.18, because it finds almost nothing.

In [ ]:
pd.Series({"accuracy": accuracy_score(y_held, predicted_classes),
           "precision": precision_score(y_held, predicted_classes),
           "recall": recall_score(y_held, predicted_classes),
           "F1": f1_score(y_held, predicted_classes)}).round(3)

`classification_report` prints these metrics for each class, with the **support**, the number of rows of the class. The macro average treats the two classes equally, and the weighted average weights them by their support.

In [ ]:
print(classification_report(y_held, predicted_classes, target_names=["benign", "malignant"]))

`DummyClassifier` is the baseline of classification: it predicts the most frequent class for every row. Its accuracy equals the share of benign tumours, and its recall of the malignant class is 0, which is why accuracy alone can mislead when the classes are unequal.

In [ ]:
dummy = DummyClassifier(strategy="most_frequent").fit(X_fit, y_fit)
dummy_predictions = dummy.predict(X_held)
round(accuracy_score(y_held, dummy_predictions), 3), recall_score(y_held, dummy_predictions)

In this screening task a missed cancer costs far more than a false alarm, which leads to another test, so **recall** is the metric that matters most, with precision beside it.

## 4. Nearest Neighbours and a Decision Tree as Classifiers

Two other families make no assumption about the shape of the boundary between the classes.

The **k-nearest-neighbour classifier** predicts a tumour from the k most similar training tumours:

1. it computes the distance $\sqrt{\sum_j (a_j - b_j)^2}$ from the new tumour to every training tumour,
2. it keeps the k training tumours with the smallest distances, and
3. it predicts the class that most of those k tumours have.

A distance adds up the differences in every column, so the columns must be on comparable scales: without scaling, a measurement in the hundreds, such as the area, decides every distance. The number of neighbours k is a hyperparameter: with k = 1 the prediction copies the single closest tumour, and with a large k it averages away real differences.

A **decision tree** asks a sequence of yes-or-no questions, each about one column, such as "is the worst radius at most 16.8?", and predicts the majority class of the group of training tumours that reach the same final answer, a **leaf**. A tree does not need scaling, because a threshold on one column ignores the units of the others. Exercise 6 opens the tree and shows how it chooses its questions.

In [ ]:
classifiers = {
    "k nearest neighbours, unscaled": KNeighborsClassifier(n_neighbors=5),
    "k nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": DecisionTreeClassifier(random_state=SEED),
}

In [ ]:
pd.DataFrame({name: {"accuracy": accuracy_score(y_held, model.fit(X_fit, y_fit).predict(X_held)),
                     "recall": recall_score(y_held, model.predict(X_held))}
              for name, model in classifiers.items()}).T.round(3)

These scores come from the test set, which a real analysis would not touch until the end. They are shown once to make the point of the next section: one split of 114 tumours gives a noisy score, and the choice between models has to rest on cross-validation of the training rows.

## 5. Naive Bayes

Bayes' theorem, which Exercise 2 used, gives the probability of a class $c$ given a row $x = (x_1, \dots, x_p)$:

$$P(c \mid x) = \frac{P(c) \, P(x \mid c)}{P(x)}$$

The denominator is the same for every class, so the class with the largest $P(c) \, P(x \mid c)$ wins. Estimating $P(x \mid c)$ for every combination of $p$ columns would need enormous data. The **naive** assumption is that the columns are independent within each class, so the joint probability becomes a product of one-column probabilities that can be counted:

$$P(c \mid x) \propto P(c) \prod_j P(x_j \mid c)$$

A label column uses the share of each label within the class, and a numerical column usually uses a normal density with the mean and the variance of the class. The ten records of `data/tax-evasion.csv` show every step by hand. `income` is in thousands of dollars.

In [ ]:
tax = pd.read_csv(DATA / "tax-evasion.csv")
tax

In [ ]:
priors = tax["evade"].value_counts(normalize=True)
refund = pd.crosstab(tax["refund"], tax["evade"], normalize="columns")
status = pd.crosstab(tax["status"], tax["evade"], normalize="columns")
income = tax.groupby("evade")["income"].agg(["mean", "var"])
priors.round(3).to_dict(), refund.round(3), status.round(3), income

The record to classify is a single taxpayer with a refund and an income of 80. The income enters through the normal density of each class, with the variance of the sample, which divides by $n_c - 1$.

In [ ]:
def normal_density(x: float, mean: float, variance: float) -> float:
    return float(np.exp(-(x - mean) ** 2 / (2 * variance)) / np.sqrt(2 * np.pi * variance))


scores = {c: priors[c] * refund.loc["yes", c] * status.loc["single", c]
          * normal_density(80, income.loc[c, "mean"], income.loc[c, "var"]) for c in ["no", "yes"]}
{c: round(float(score), 6) for c, score in scores.items()}

No evader in the table had a refund, so $P(\text{refund} = \text{yes} \mid \text{yes}) = 0$, and that one zero makes the score of the class 0 whatever the other columns say. **Laplace smoothing** adds 1 to every count, so the share of a label becomes $(\text{count} + 1) / (n_c + k)$ for a column with $k$ labels, and no estimate is ever exactly 0.

In [ ]:
def smoothed(column: str, value: str, c: str) -> float:
    in_class = tax[tax["evade"] == c]
    return float(((in_class[column] == value).sum() + 1) / (len(in_class) + tax[column].nunique()))


smooth = {c: priors[c] * smoothed("refund", "yes", c) * smoothed("status", "single", c)
          * normal_density(80, income.loc[c, "mean"], income.loc[c, "var"]) for c in ["no", "yes"]}
{c: round(float(score), 6) for c, score in smooth.items()}, round(smooth["no"] / sum(smooth.values()), 3)

The decision is still `no`, but it is no longer forced by a single zero. Dividing the two scores by their sum gives the probability of each class under the model. Products of many small numbers underflow, so implementations add logarithms instead of multiplying probabilities.

Scikit-learn offers one naive Bayes classifier for each kind of column: `GaussianNB` for numbers, `CategoricalNB` for labels, `BernoulliNB` for 0-or-1 columns, and `MultinomialNB` for counts. `GaussianNB` on the 30 tumour measurements, scaled first, is cross-validated on the training tumours like the other classifiers.

In [ ]:
gaussian = make_pipeline(StandardScaler(), GaussianNB())
cross_val_score(gaussian, X_fit, y_fit, cv=StratifiedKFold(5, shuffle=True, random_state=SEED)).mean().round(3)

Scaled and cross-validated on the training tumours, Gaussian naive Bayes reaches an accuracy of 0.93, below the logistic regression of Section 2. Many of the measurements, such as the radius, the perimeter, and the area of a nucleus, are strongly correlated within each class, so the independence assumption is far from true and the same evidence is counted several times. The model learns only a mean and a variance for every column in every class, stored in `theta_` and `var_`, which makes it very fast to fit.

## 6. Cross-Validation for Classifiers

Exercise 4 introduced k-fold cross-validation. For classification, **stratified** folds keep the class shares of the whole training set in every fold. `StratifiedKFold` defines them, and `cross_validate` scores several metrics on each fold.

In [ ]:
strata = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
small_tree = DecisionTreeClassifier(criterion="entropy", max_depth=3, min_samples_leaf=3, random_state=SEED)
tree_scores = cross_validate(small_tree, X_fit, y_fit, scoring=["precision", "recall", "f1"], cv=strata)
pd.DataFrame(tree_scores).drop(columns=["fit_time", "score_time"]).round(3)

In [ ]:
round(tree_scores["test_f1"].mean(), 3), round(tree_scores["test_f1"].std(), 3)

### How Much a Score Moves

A single held-out split gives a score that depends on which rows land in it. The logistic regression is scored on 200 random stratified 80/20 splits of all 569 tumours, and then by 5-fold cross-validation repeated with 200 different shuffles.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression())
holdout = []
for seed in range(200):
    X_tr, X_te, y_tr, y_te = train_test_split(X_cells, y_cells, test_size=0.2, stratify=y_cells,
                                              random_state=seed)
    holdout.append(accuracy_score(y_te, model.fit(X_tr, y_tr).predict(X_te)))
repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=200, random_state=SEED)
fold_accuracy = cross_val_score(model, X_cells, y_cells, cv=repeated, scoring="accuracy")
fold_accuracy = fold_accuracy.reshape(200, 5).mean(axis=1)
spread = pd.DataFrame({"one 80/20 split": holdout, "5-fold mean": fold_accuracy}).describe()
spread.loc[["mean", "std", "min", "max"]].round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.hist(holdout, bins=20, alpha=0.7, color=PALETTE[1], label="one 80/20 split")
ax.hist(fold_accuracy, bins=20, alpha=0.7, color=PALETTE[0], label="mean of 5 folds")
ax.set(xlabel="Accuracy", ylabel="Shuffles", title="200 hold-out splits against 200 cross-validations")
ax.legend()
plt.show()

The same model scores anywhere from about 0.93 to 1.00 on a single split, while the mean of five folds varies far less. The spread measures how much a score depends on the split of this dataset, not how it would vary on new datasets.

### Choosing the Number of Folds

A small k trains each model on fewer rows, which makes the estimate pessimistic, and a large k costs more fits and leaves small validation folds with noisy scores. Values of 5 or 10 are a common balance. **Leave-one-out** cross-validation, with k equal to the number of rows, is the extreme case.

In [ ]:
rows = {}
for k in [2, 3, 5, 10, 20]:
    k_scores = cross_val_score(model, X_fit, y_fit, cv=StratifiedKFold(k, shuffle=True, random_state=SEED))
    rows[k] = {"mean accuracy": k_scores.mean(), "fold std": k_scores.std()}
pd.DataFrame(rows).round(4)

### Splits That Follow the Data

A random split assumes that the rows are independent. Two common kinds of data break that assumption, and each has its own splitter:

- **grouped** rows, such as several images of the same patient, must stay together, or the model is scored on a patient it has already seen. `GroupKFold` keeps every group in one fold.
- **ordered** rows, such as daily sales, must be split in time, or the model is scored on the past after training on the future. `TimeSeriesSplit` always trains on earlier rows and validates on later ones.

The splitters are shown on twelve rows that belong to four groups.

In [ ]:
rows_12 = np.arange(12)
groups = np.repeat(["A", "B", "C", "D"], 3)
time_splits = list(TimeSeriesSplit(n_splits=4).split(rows_12))
pd.DataFrame({
    "grouped": ["".join(groups[held]) for _, held in GroupKFold(n_splits=4).split(rows_12, groups=groups)],
    "time: train": [f"{fit.min()}-{fit.max()}" for fit, _ in time_splits],
    "time: validate": [f"{held.min()}-{held.max()}" for _, held in time_splits],
}, index=[f"split {i + 1}" for i in range(4)])

## 7. Tuning Hyperparameters

A hyperparameter is chosen by comparing held-out scores. A **validation curve** draws the training and validation scores against one hyperparameter. `validation_curve` cross-validates the decision tree for every maximum depth from 1 to 12.

In [ ]:
depth_range = np.arange(1, 13)
train_scores, valid_scores = validation_curve(DecisionTreeClassifier(random_state=SEED), X_fit, y_fit,
                                              param_name="max_depth", param_range=depth_range, cv=strata,
                                              scoring="accuracy")
fig, ax = plt.subplots(figsize=(9, 3.4))
ax.plot(depth_range, train_scores.mean(axis=1), marker="o", label="training")
ax.plot(depth_range, valid_scores.mean(axis=1), marker="o", label="validation (5 folds)")
ax.set(xlabel="max_depth", ylabel="Accuracy", title="Validation curve of a decision tree",
       xticks=depth_range)
ax.legend()
plt.show()
int(depth_range[valid_scores.mean(axis=1).argmax()]), round(valid_scores.mean(axis=1).max(), 3)

The training accuracy reaches 1 once the tree is deep enough to put every tumour in a pure leaf, while the validation accuracy stops improving after a few levels.

### Grid Search and Random Search

Most models have several hyperparameters. **Grid search** tries every combination of the values listed in a grid and cross-validates each one. `GridSearchCV` does this and then refits the best combination on all the rows it received, so the fitted search behaves like a model. The grid below gives 3 criteria, 6 depths, and 4 minimum split sizes, so 72 combinations of 5 fits each, 360 fits in all. The cost grows by multiplication: one more hyperparameter with 5 values would make 1,800 fits. `scoring="recall"` makes the search prefer the tree that misses the fewest malignant tumours, the metric chosen in Section 3.

In [ ]:
parameters = {
    "criterion": ["gini", "entropy", "log_loss"],
    "max_depth": [None, 1, 2, 3, 4, 5],
    "min_samples_split": [2, 3, 4, 5],
}
grid_search = GridSearchCV(DecisionTreeClassifier(random_state=SEED), param_grid=parameters, cv=strata,
                           scoring="recall")
grid_search.fit(X_fit, y_fit)
grid_search.best_params_, round(grid_search.best_score_, 3)

**Random search** samples a fixed number of combinations from distributions instead. When only a few hyperparameters matter, it tries more distinct values of each for the same budget. `RandomizedSearchCV` draws 20 values of `C` for the logistic regression from a log-uniform distribution between 0.001 and 100. A hyperparameter of a pipeline step is named after the step, two underscores, and the parameter.

In [ ]:
random_search = RandomizedSearchCV(make_pipeline(StandardScaler(), LogisticRegression()),
                                   param_distributions={"logisticregression__C": stats.loguniform(1e-3, 100)},
                                   n_iter=20, cv=strata, scoring="recall", random_state=SEED)
random_search.fit(X_fit, y_fit)
round(random_search.best_params_["logisticregression__C"], 3), round(random_search.best_score_, 3)

The best cross-validated score of a search is optimistic, because it is the largest of many noisy scores. It chooses the hyperparameters, and the test set measures the result.

## 8. Choosing the Classifier

The candidates are compared on the same stratified folds and the same metrics, so that their scores differ only because the models differ.

In [ ]:
candidates = {
    "most frequent class": DummyClassifier(strategy="most_frequent"),
    "Gaussian naive Bayes": make_pipeline(StandardScaler(), GaussianNB()),
    "logistic regression": random_search.best_estimator_,
    "k nearest neighbours": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "decision tree": grid_search.best_estimator_,
}

In [ ]:
comparison = pd.DataFrame({
    name: {metric: cross_val_score(model, X_fit, y_fit, cv=strata, scoring=metric).mean()
           for metric in ["accuracy", "precision", "recall", "f1"]}
    for name, model in candidates.items()}).T
comparison.round(3)

Gaussian naive Bayes is clearly weaker than the other three classifiers. The logistic regression has the highest accuracy and F1, and it shares the highest recall, the metric chosen in Section 3, with the tree, which raises far more false alarms. Nearest neighbours has the highest precision but misses more cancers. The search has already refitted the chosen logistic regression on all 455 training tumours, and the test set is used once.

In [ ]:
chosen = random_search.best_estimator_
print(classification_report(y_held, chosen.predict(X_held), target_names=["benign", "malignant"]))

On the 114 test tumours, the chosen model finds almost every malignant tumour, with few false alarms.

## 9. Thresholds, ROC, and Precision-Recall Curves

The rest of the exercise evaluates classifiers in more depth. `data/german-credit.csv` describes 1,000 loan applicants by 20 columns, such as the status of their checking account, the duration and the amount of the loan, and their age, and `risk` says whether each turned out a good or a bad credit risk. A bad risk, a customer who does not repay, is the case to detect, so it is the positive class 1.

In [ ]:
credit = pd.read_csv(DATA / "german-credit.csv")
X_credit = credit.drop(columns="risk")
y_credit = (credit["risk"] == "bad").astype(int)
X_credit_fit, X_credit_test, y_credit_fit, y_credit_test = train_test_split(
    X_credit, y_credit, test_size=0.25, stratify=y_credit, random_state=SEED)
credit.shape, round(y_credit.mean(), 3), len(X_credit_fit), len(X_credit_test)

In [ ]:
credit_labels = list(X_credit.select_dtypes(exclude="number").columns)
credit_numbers = list(X_credit.select_dtypes(include="number").columns)
credit_preparation = ColumnTransformer([
    ("labels", OneHotEncoder(handle_unknown="ignore"), credit_labels),
    ("numbers", StandardScaler(), credit_numbers),
])
credit_model = make_pipeline(credit_preparation, LogisticRegression(max_iter=1000))
credit_model.fit(X_credit_fit, y_credit_fit)
credit_predictions = credit_model.predict(X_credit_test)
confusion_matrix(y_credit_test, credit_predictions)

The model misses 37 of the 75 bad customers in the test set, because `predict` flags a customer only when the probability of a bad risk is at least 0.5. A probability becomes a decision at a **threshold**: a customer is flagged when $P(\text{bad}) \ge t$. Lowering $t$ flags more customers, which raises the recall and usually lowers the precision. To choose a threshold without touching the test set, every training customer receives an **out-of-fold** probability from `cross_val_predict`: each fold is predicted by a model fitted on the other folds.

In [ ]:
out_of_fold = cross_val_predict(credit_model, X_credit_fit, y_credit_fit, cv=strata,
                                method="predict_proba")[:, 1]
thresholds = np.arange(0.1, 0.91, 0.2)
good = (y_credit_fit == 0).to_numpy()
pd.DataFrame({t: {"precision": precision_score(y_credit_fit, out_of_fold >= t, zero_division=0),
                  "recall": recall_score(y_credit_fit, out_of_fold >= t),
                  "false positive rate": (out_of_fold[good] >= t).mean(),
                  "F1": f1_score(y_credit_fit, out_of_fold >= t)} for t in thresholds.round(1)}).round(3)

A picture makes the trade-off concrete. The two histograms show the out-of-fold probabilities of the good and the bad customers. A threshold is a vertical line: every customer to its right is flagged. Moving the line to the left flags more of the bad customers, which raises the recall, but also more of the good ones, which raises the false positive rate. The two groups overlap, so no line separates them perfectly.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 3.6))
bins = np.linspace(0, 1, 26)
ax.hist(out_of_fold[y_credit_fit == 0], bins=bins, alpha=0.6, color=PALETTE[0], label="good customers")
ax.hist(out_of_fold[y_credit_fit == 1], bins=bins, alpha=0.6, color=PALETTE[1], label="bad customers")
ax.axvline(0.5, color=INK, linestyle="--", label="threshold 0.50")
ax.set(xlabel="Out-of-fold probability of a bad risk", ylabel="Customers", title="Two overlapping groups")
ax.legend()
plt.show()

The table reads column by column. At the default threshold of 0.5, the model finds 47.6 % of the bad customers and wrongly flags 13.5 % of the good ones. At a threshold of 0.1, it finds 91.6 % of the bad customers but flags 63.2 % of the good ones. Each column is one possible decision rule, and the next curves draw all of them at once.

Two curves summarize every threshold at once:

- the **ROC curve**, short for receiver operating characteristic, draws the recall, also called the true positive rate, against the **false positive rate** $FP / (FP + TN)$. The area under it, the **ROC AUC**, is the probability that a random bad customer gets a higher score than a random good one. A random ranking scores 0.5 and a perfect one 1.
- the **precision-recall curve** draws the precision against the recall. Its summary, the **average precision**, starts from the share of positives, which a random ranking reaches, so it shows more clearly how well a model finds a rare class.

In [ ]:
fpr, tpr, _ = roc_curve(y_credit_fit, out_of_fold)
precision_curve, recall_curve, _ = precision_recall_curve(y_credit_fit, out_of_fold)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fpr, tpr, color=PALETTE[0])
axes[0].plot([0, 1], [0, 1], color=GRID, linestyle="--")
axes[0].set(xlabel="False positive rate", ylabel="True positive rate (recall)",
            title=f"ROC curve, AUC {roc_auc_score(y_credit_fit, out_of_fold):.3f}")
axes[1].plot(recall_curve, precision_curve, color=PALETTE[1])
axes[1].axhline(y_credit_fit.mean(), color=GRID, linestyle="--")
axes[1].set(xlabel="Recall", ylabel="Precision",
            title=f"Precision-recall curve, AP {average_precision_score(y_credit_fit, out_of_fold):.3f}")
plt.show()

The dashed lines are what a random ranking reaches. The AUC measures ranking only: it says nothing about whether the probabilities are right or which threshold to use.

## 10. Class Imbalance

A class is **imbalanced** when it is much rarer than the other, as fraud or a rare disease is. Here 30 % of the customers are bad, a mild imbalance. A model fitted to imbalanced data leans towards the frequent class, and three remedies push it back:

- **class weights**, `class_weight="balanced"`, keep every row but multiply the loss of each class by $n / (K n_k)$ for class $k$ of $K$, so an error on the rare class costs more,
- **under-sampling** discards rows of the frequent class, and
- **over-sampling** repeats rows of the rare class, or synthesizes new ones: **SMOTE** draws a new row on the line between a rare row and one of its rare nearest neighbours.

The picture below shows SMOTE on a small two-dimensional example with 21 rare points among 199 frequent ones. Each new point lies on a segment between a rare point and one of its rare neighbours, so the rare class is filled in rather than copied.

In [ ]:
from sklearn.datasets import make_classification

X_small, y_small = make_classification(n_samples=220, n_features=2, n_redundant=0, n_clusters_per_class=1,
                                       weights=[0.9], class_sep=1.2, random_state=SEED)
X_smote, y_smote = SMOTE(k_neighbors=3, random_state=SEED).fit_resample(X_small, y_small)
synthetic = X_smote[len(X_small):]
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True, sharey=True)
for ax, title in zip(axes, ["original", "after SMOTE"]):
    ax.scatter(*X_small[y_small == 0].T, s=10, color=GRID, label="frequent class")
    ax.scatter(*X_small[y_small == 1].T, s=30, color=PALETTE[1], label="rare class")
    ax.set(title=title, xlabel="x1")
axes[1].scatter(*synthetic.T, s=18, marker="x", color=PALETTE[0], label="synthetic rare rows")
axes[0].set_ylabel("x2")
axes[1].legend(loc="lower right")
plt.show()
np.bincount(y_small), np.bincount(y_smote)

SMOTE adds rare rows until the two classes are equal in size. Resampling must happen inside the training folds only. Over-sampling before the split would copy a rare row into both the training and the validation folds, which is leakage. The package `imbalanced-learn` provides samplers and a pipeline that applies them during `fit` and skips them during `predict`.

In [ ]:
balancing = {
    "no remedy": credit_model,
    "class weights": make_pipeline(credit_preparation,
                                   LogisticRegression(class_weight="balanced", max_iter=1000)),
    "random over-sampling": make_imbalanced_pipeline(
        credit_preparation, RandomOverSampler(random_state=SEED), LogisticRegression(max_iter=1000)),
    "random under-sampling": make_imbalanced_pipeline(
        credit_preparation, RandomUnderSampler(random_state=SEED), LogisticRegression(max_iter=1000)),
    "SMOTE": make_imbalanced_pipeline(credit_preparation, SMOTE(random_state=SEED),
                                      LogisticRegression(max_iter=1000)),
}
metrics = ["precision", "recall", "f1", "average_precision"]
remedies = {}
for name, model in balancing.items():
    folds = cross_validate(model, X_credit_fit, y_credit_fit, cv=strata, scoring=metrics)
    remedies[name] = {m: folds[f"test_{m}"].mean() for m in metrics}
pd.DataFrame(remedies).T.round(3)

Every remedy trades precision for recall in much the same way as lowering the threshold does, and none changes the average precision much, because the ranking of the customers barely changes. When a model gives probabilities, moving the threshold is the simpler remedy. Resampling and weights matter more when the rare class is very rare, or when a model has no probabilities to move a threshold on.

### Summary

- Logistic regression turned log-odds into a probability, and the confusion matrix turned decisions into precision, recall, and F1.
- Nearest neighbours, a decision tree, and naive Bayes were compared with a baseline on the same stratified folds, inside pipelines that refit every preparation step.
- Naive Bayes counted, assumed independence within each class, and needed Laplace smoothing against zero counts.
- Hyperparameters were tuned by grid search and random search, and the test set measured the chosen model once.
- A threshold turns a probability into a decision, and moving it trades missed positives for false alarms.
- Curves summarize every threshold, and resampling and class weights counter imbalance.